<a href="https://colab.research.google.com/github/Usha-Shatul/DL/blob/main/DL_1st_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from google.colab import drive

# =========================
# LOAD DATASET
# =========================
drive.mount('/content/drive')

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/dataset/Breast Cancer Wisconsin Dataset.csv")
print("Dataset loaded from Google Drive")
print("Dataset Head:")
print(df.head())
print(f"\nDataset Shape: {df.shape}")

# =========================
# CLEAN DATASET
# =========================

# Drop 'id' column if exists
if "id" in df.columns:
    df = df.drop("id", axis=1)

# Drop 'Unnamed: 32' column if exists (contains only NaN)
if "Unnamed: 32" in df.columns:
    df = df.drop("Unnamed: 32", axis=1)

# Convert label: M = 1, B = 0
if "diagnosis" in df.columns and df["diagnosis"].dtype == 'object':
    df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

# Remove any rows with missing values
df = df.dropna()

print(f"\nDataset after cleaning: {df.shape}")

# =========================
# FEATURES & LABEL
# =========================
X = df.drop("diagnosis", axis=1).values
Y = df["diagnosis"].values

# Check if dataset is empty
if len(X) == 0:
    raise ValueError("Dataset is empty! Please check your data file.")

print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"Class distribution: Benign={(Y==0).sum()}, Malignant={(Y==1).sum()}")

# Normalize (VERY IMPORTANT for NN)
X = (X - X.mean(axis=0)) / X.std(axis=0)

# Use only first feature for single neuron
X_single = X[:, 0]

# =====================================================
# SECTION 1: SINGLE NEURON MODEL (x1 only)
# =====================================================

print("\n" + "="*60)
print("SECTION 1: SINGLE NEURON MODEL (Using only first feature)")
print("="*60)

w = 0.2
b = 0.0
lr = 0.01
epochs_single = 200

# Store predictions for each epoch to track accuracy
single_epoch_accuracies = []

for epoch in range(epochs_single):
    total_loss = 0
    epoch_preds = []

    for i in range(len(X_single)):
        y_pred = w * X_single[i] + b

        error = y_pred - Y[i]
        total_loss += error ** 2

        w -= lr * (2 * error * X_single[i])
        b -= lr * (2 * error)

    # Calculate predictions for this epoch
    epoch_preds = []
    for i in range(len(X_single)):
        pred = w * X_single[i] + b
        epoch_preds.append(1 if pred >= 0.5 else 0)

    # Calculate accuracy for this epoch (manually)
    correct = sum(1 for i in range(len(Y)) if epoch_preds[i] == Y[i])
    epoch_accuracy = correct / len(Y)
    single_epoch_accuracies.append(epoch_accuracy)

    # Store final predictions
    if epoch == epochs_single - 1:
        single_preds = epoch_preds

    # Print accuracy every 20 epochs
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss={total_loss:.4f} | Accuracy={epoch_accuracy:.4f} ({epoch_accuracy*100:.2f}%)")

print(f"\nFinal Single Neuron - W: {w:.4f}, B: {b:.4f}")
print(f"Final Accuracy: {single_epoch_accuracies[-1]:.4f} ({single_epoch_accuracies[-1]*100:.2f}%)")

# =====================================================
# SECTION 2: NEURAL NETWORK MODEL (2 hidden neurons)
# =====================================================

print("\n" + "="*60)
print("SECTION 2: NEURAL NETWORK MODEL (2 Hidden Neurons)")
print("="*60)

# input size = number of features
input_size = X.shape[1]

# Initialize weights with small random values
np.random.seed(42)
W1 = np.random.randn(2, input_size) * 0.01
b1 = np.zeros(2)
W2 = np.random.randn(2) * 0.01
b2 = 0.0

lr = 0.1
epochs_nn = 200

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -250, 250)))  # Clip to prevent overflow

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Store predictions for each epoch to track accuracy
nn_epoch_accuracies = []

for epoch in range(epochs_nn):
    total_loss = 0
    epoch_preds = []

    for i in range(len(X)):
        # =========================
        # Forward pass
        # =========================
        # Hidden layer
        z1 = np.dot(W1, X[i]) + b1
        h = sigmoid(z1)

        # Output layer
        z2 = np.dot(W2, h) + b2
        y_pred = sigmoid(z2)

        # Store prediction
        epoch_preds.append(1 if y_pred >= 0.5 else 0)

        # =========================
        # Backpropagation
        # =========================
        error = y_pred - Y[i]
        total_loss += error ** 2

        # Output layer gradient
        d_output = error * sigmoid_derivative(z2)

        # Hidden layer gradient
        d_hidden = d_output * W2 * sigmoid_derivative(z1)

        # Update weights and biases
        W2 -= lr * d_output * h
        b2 -= lr * d_output

        # Update W1 and b1
        W1 -= lr * np.outer(d_hidden, X[i])
        b1 -= lr * d_hidden

    # Calculate accuracy for this epoch (manually)
    correct = sum(1 for i in range(len(Y)) if epoch_preds[i] == Y[i])
    epoch_accuracy = correct / len(Y)
    nn_epoch_accuracies.append(epoch_accuracy)

    # Store final predictions
    if epoch == epochs_nn - 1:
        nn_preds = epoch_preds

    # Print accuracy every 20 epochs
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss={total_loss:.4f} | Accuracy={epoch_accuracy:.4f} ({epoch_accuracy*100:.2f}%)")

print(f"\nFinal Neural Network - W2: {W2}")
print(f"Final Accuracy: {nn_epoch_accuracies[-1]:.4f} ({nn_epoch_accuracies[-1]*100:.2f}%)")

# =====================================================
# SECTION 3: MODEL EVALUATION (MANUAL METRICS - NO VISUALIZATION)
# =====================================================

print("\n" + "="*60)
print("SECTION 3: MODEL EVALUATION")
print("="*60)

def calculate_confusion_matrix(y_true, y_pred):
    """Calculate confusion matrix manually"""
    cm = np.zeros((2, 2), dtype=int)

    for i in range(len(y_true)):
        actual = int(y_true[i])
        predicted = int(y_pred[i])
        cm[actual][predicted] += 1

    return cm

def print_confusion_matrix(cm, model_name):
    """Print confusion matrix in text format"""
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]

    print(f"\n{model_name} - Confusion Matrix:")
    print("="*50)
    print("                 Predicted")
    print("                 Benign  Malignant")
    print(f"  Actual Benign    {tn:3d}      {fp:3d}")
    print(f"         Malignant  {fn:3d}      {tp:3d}")
    print("="*50)

    return tn, fp, fn, tp

def calculate_metrics_manual(y_true, y_pred, model_name):
    """Calculate all metrics manually without sklearn and without visualization"""
    print(f"\n--- {model_name} ---")

    # Check if predictions are valid
    if len(y_pred) == 0:
        print("Warning: No predictions available!")
        return {
            'accuracy': 0,
            'precision': 0,
            'recall': 0,
            'f1': 0,
            'specificity': 0,
            'confusion_matrix': np.array([[0, 0], [0, 0]]),
            'tp': 0, 'tn': 0, 'fp': 0, 'fn': 0
        }

    # Calculate Confusion Matrix
    cm = calculate_confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = print_confusion_matrix(cm, model_name)

    # Calculate Accuracy
    total = tn + fp + fn + tp
    if total == 0:
        accuracy = 0
    else:
        accuracy = (tp + tn) / total
    print(f"\nAccuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

    # Calculate Precision
    if (tp + fp) == 0:
        precision = 0
    else:
        precision = tp / (tp + fp)
    print(f"Precision: {precision:.4f}")

    # Calculate Recall (Sensitivity)
    if (tp + fn) == 0:
        recall = 0
    else:
        recall = tp / (tp + fn)
    print(f"Recall (Sensitivity): {recall:.4f}")

    # Calculate Specificity
    if (tn + fp) == 0:
        specificity = 0
    else:
        specificity = tn / (tn + fp)
    print(f"Specificity: {specificity:.4f}")

    # Calculate F1-Score
    if (precision + recall) == 0:
        f1 = 0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
    print(f"F1-Score: {f1:.4f}")

    # Print detailed classification report
    print(f"\nClassification Report:")
    print("-"*60)
    print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
    print("-"*60)

    # Benign class metrics
    benign_precision = tn / (tn + fn) if (tn + fn) > 0 else 0
    benign_recall = tn / (tn + fp) if (tn + fp) > 0 else 0
    benign_f1 = 2 * (benign_precision * benign_recall) / (benign_precision + benign_recall) if (benign_precision + benign_recall) > 0 else 0
    benign_support = tn + fn

    # Malignant class metrics
    malignant_precision = precision
    malignant_recall = recall
    malignant_f1 = f1
    malignant_support = tp + fn

    print(f"{'Benign (0)':<12} {benign_precision:>10.3f} {benign_recall:>10.3f} {benign_f1:>10.3f} {benign_support:>10d}")
    print(f"{'Malignant (1)':<12} {malignant_precision:>10.3f} {malignant_recall:>10.3f} {malignant_f1:>10.3f} {malignant_support:>10d}")
    print("-"*60)

    # Macro average
    macro_precision = (benign_precision + malignant_precision) / 2
    macro_recall = (benign_recall + malignant_recall) / 2
    macro_f1 = (benign_f1 + malignant_f1) / 2
    print(f"{'Macro Avg':<12} {macro_precision:>10.3f} {macro_recall:>10.3f} {macro_f1:>10.3f} {total:>10d}")
    print("-"*60)
    print(f"{'Accuracy':<12} {accuracy:>33.4f}")
    print("="*60)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'confusion_matrix': cm,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn
    }

# Calculate metrics for Single Neuron
print("\n" + "-"*40)
print("SINGLE NEURON MODEL - Detailed Results")
print("-"*40)
single_metrics = calculate_metrics_manual(Y, single_preds, "Single Neuron")

# Calculate metrics for Neural Network
print("\n" + "-"*40)
print("NEURAL NETWORK MODEL - Detailed Results")
print("-"*40)
nn_metrics = calculate_metrics_manual(Y, nn_preds, "Neural Network")

# =====================================================
# SECTION 4: MODEL COMPARISON
# =====================================================

print("\n" + "="*60)
print("SECTION 4: MODEL COMPARISON")
print("="*60)

print("\nComparison Table:")
print("="*70)
print(f"{'Metric':<15} {'Single Neuron':>20} {'Neural Network':>20}")
print("-"*70)
print(f"{'Accuracy':<15} {single_metrics['accuracy']:>20.4f} {nn_metrics['accuracy']:>20.4f}")
print(f"{'Precision':<15} {single_metrics['precision']:>20.4f} {nn_metrics['precision']:>20.4f}")
print(f"{'Recall':<15} {single_metrics['recall']:>20.4f} {nn_metrics['recall']:>20.4f}")
print(f"{'Specificity':<15} {single_metrics['specificity']:>20.4f} {nn_metrics['specificity']:>20.4f}")
print(f"{'F1-Score':<15} {single_metrics['f1']:>20.4f} {nn_metrics['f1']:>20.4f}")
print("="*70)

# =====================================================
# SECTION 5: EPOCH-WISE ACCURACY COMPARISON
# =====================================================

print("\n" + "="*60)
print("SECTION 5: EPOCH-WISE ACCURACY COMPARISON")
print("="*60)

print("\nAccuracy at every 20th epoch:")
print("="*60)
print(f"{'Epoch':>10} {'Single Neuron':>20} {'Neural Network':>20}")
print("-"*60)

for i in range(0, epochs_single, 20):
    single_acc = single_epoch_accuracies[i] if i < len(single_epoch_accuracies) else 0
    nn_acc = nn_epoch_accuracies[i] if i < len(nn_epoch_accuracies) else 0
    print(f"{i:>10d} {single_acc:>20.4f} {nn_acc:>20.4f}")
print("="*60)

# Find best accuracy and epoch
best_single_epoch = max(range(len(single_epoch_accuracies)), key=lambda i: single_epoch_accuracies[i])
best_nn_epoch = max(range(len(nn_epoch_accuracies)), key=lambda i: nn_epoch_accuracies[i])

print(f"\nBest Single Neuron Accuracy: {single_epoch_accuracies[best_single_epoch]:.4f} at epoch {best_single_epoch}")
print(f"Best Neural Network Accuracy: {nn_epoch_accuracies[best_nn_epoch]:.4f} at epoch {best_nn_epoch}")

# =====================================================
# SECTION 6: SAMPLE PREDICTIONS
# =====================================================

print("\n" + "="*60)
print("SECTION 6: SAMPLE PREDICTIONS")
print("="*60)

num_samples = min(10, len(X))
print(f"\nPredictions for first {num_samples} samples:")
print("="*70)
print(f"{'Index':>6} {'Actual':>8} {'Single Neuron':>15} {'Neural Network':>17} {'Correct?':>10}")
print("-"*70)

for i in range(num_samples):
    correct_single = "✓" if single_preds[i] == Y[i] else "✗"
    correct_nn = "✓" if nn_preds[i] == Y[i] else "✗"
    print(f"{i:>6d} {Y[i]:>8d} {single_preds[i]:>15d} {nn_preds[i]:>17d} {correct_single}/{correct_nn:>7}")
print("="*70)

# =====================================================
# SECTION 7: ADDITIONAL ANALYSIS
# =====================================================

print("\n" + "="*60)
print("SECTION 7: ADDITIONAL ANALYSIS")
print("="*60)

# Calculate how many samples each model got right
single_correct = sum(1 for i in range(len(Y)) if single_preds[i] == Y[i])
nn_correct = sum(1 for i in range(len(Y)) if nn_preds[i] == Y[i])

print(f"\nSingle Neuron - Correct Predictions: {single_correct}/{len(Y)} ({single_correct/len(Y)*100:.2f}%)")
print(f"Neural Network - Correct Predictions: {nn_correct}/{len(Y)} ({nn_correct/len(Y)*100:.2f}%)")

# Error analysis
single_errors = [i for i in range(len(Y)) if single_preds[i] != Y[i]]
nn_errors = [i for i in range(len(Y)) if nn_preds[i] != Y[i]]

if single_errors:
    print(f"\nSingle Neuron Misclassified indices: {single_errors[:10]}... (showing first 10)")
if nn_errors:
    print(f"Neural Network Misclassified indices: {nn_errors[:10]}... (showing first 10)")

# =====================================================
# SECTION 8: CONFUSION MATRIX DETAILS
# =====================================================

print("\n" + "="*60)
print("SECTION 8: CONFUSION MATRIX DETAILS")
print("="*60)

print("\nSingle Neuron Confusion Matrix Details:")
print(f"  True Negatives (TN): {single_metrics['tn']}")
print(f"  False Positives (FP): {single_metrics['fp']}")
print(f"  False Negatives (FN): {single_metrics['fn']}")
print(f"  True Positives (TP): {single_metrics['tp']}")

print("\nNeural Network Confusion Matrix Details:")
print(f"  True Negatives (TN): {nn_metrics['tn']}")
print(f"  False Positives (FP): {nn_metrics['fp']}")
print(f"  False Negatives (FN): {nn_metrics['fn']}")
print(f"  True Positives (TP): {nn_metrics['tp']}")

# =====================================================
# SECTION 9: PERFORMANCE COMPARISON TABLE
# =====================================================

print("\n" + "="*60)
print("SECTION 9: PERFORMANCE COMPARISON SUMMARY")
print("="*60)

print("\nPerformance Comparison:")
print("="*80)
print(f"{'Metric':<20} {'Single Neuron':>25} {'Neural Network':>25} {'Difference':>15}")
print("-"*80)

metrics_to_compare = ['accuracy', 'precision', 'recall', 'specificity', 'f1']
for metric in metrics_to_compare:
    single_val = single_metrics[metric]
    nn_val = nn_metrics[metric]
    diff = nn_val - single_val
    print(f"{metric.capitalize():<20} {single_val:>25.4f} {nn_val:>25.4f} {diff:>15.4f}")
print("="*80)

# =====================================================
# SECTION 10: FINAL SUMMARY
# =====================================================

print("\n" + "="*60)
print("SECTION 10: FINAL SUMMARY")
print("="*60)

# Summary
print("\n📊 Summary:")
if nn_metrics['accuracy'] > single_metrics['accuracy']:
    print(f"• Neural Network outperforms Single Neuron")
    print(f"• Accuracy improvement: {(nn_metrics['accuracy'] - single_metrics['accuracy'])*100:.2f}%")
    print(f"• F1-Score improvement: {(nn_metrics['f1'] - single_metrics['f1'])*100:.2f}%")

    # Which class improved more?
    if (nn_metrics['recall'] - single_metrics['recall']) > (nn_metrics['specificity'] - single_metrics['specificity']):
        print(f"• Main improvement: Better at detecting Malignant cases")
    else:
        print(f"• Main improvement: Better at detecting Benign cases")
else:
    print(f"• Single Neuron outperforms Neural Network")
    print(f"• Accuracy difference: {(single_metrics['accuracy'] - nn_metrics['accuracy'])*100:.2f}%")
    print(f"• F1-Score difference: {(single_metrics['f1'] - nn_metrics['f1'])*100:.2f}%")

# Dataset statistics
print(f"\n📊 Dataset Statistics:")
print(f"• Total samples: {len(Y)}")
print(f"• Benign (0): {(Y==0).sum()} ({(Y==0).sum()/len(Y)*100:.2f}%)")
print(f"• Malignant (1): {(Y==1).sum()} ({(Y==1).sum()/len(Y)*100:.2f}%)")

# Performance summary
print(f"\n🎯 Performance Summary:")
print(f"• Single Neuron Model: {single_metrics['accuracy']*100:.2f}% accuracy, {single_metrics['f1']*100:.2f}% F1-Score")
print(f"• Neural Network Model: {nn_metrics['accuracy']*100:.2f}% accuracy, {nn_metrics['f1']*100:.2f}% F1-Score")

# Recommendation
print(f"\n💡 Recommendation:")
if nn_metrics['accuracy'] > single_metrics['accuracy']:
    print(f"• Use Neural Network for better performance")
    if nn_metrics['accuracy'] - single_metrics['accuracy'] > 0.05:
        print(f"• Significant improvement (>5%), definitely use Neural Network")
    else:
        print(f"• Minor improvement, Single Neuron might be sufficient for simpler applications")
else:
    print(f"• Single Neuron is performing better, use it for this dataset")

print("\n" + "="*60)
print("=== ANALYSIS COMPLETE ===")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded from Google Drive
Dataset Head:
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390     